In [4]:
# !pip install boto3

In [19]:
import boto3
import json
import pandas as pd

In [81]:
coop_server = 'i-0b7aa9c4f1cb5e2f6'

In [67]:
ec2 = boto3.client('ec2')

In [171]:
ec2.describe_instance_status(InstanceIds = [coop_server], IncludeAllInstances = True)['InstanceStatuses']

[{'AvailabilityZone': 'us-east-2a',
  'Operator': {'Managed': False},
  'InstanceId': 'i-0b7aa9c4f1cb5e2f6',
  'InstanceState': {'Code': 80, 'Name': 'stopped'},
  'InstanceStatus': {'Status': 'not-applicable'},
  'SystemStatus': {'Status': 'not-applicable'}}]

In [141]:
def get_server_status(instance_id):
    ec2_response = ec2.describe_instance_status(InstanceIds = [instance_id], IncludeAllInstances = True)
    ec2_df = pd.DataFrame(ec2_response['InstanceStatuses'])
    ec2_df['InstanceStateCode'] = ec2_df['InstanceState'].apply(lambda x: x['Name'])
    ec2_df['InstanceStateCode'] = ec2_df['InstanceState'].apply(lambda x: x['Name'])
    server_status = ec2_df[ec2_df['InstanceId'] == coop_server]['InstanceStateCode'][0]

    return server_status

In [142]:
get_server_status(coop_server)

'stopped'

In [117]:
lmbd = boto3.client('lambda')

In [120]:
lmbd_df = pd.DataFrame(lmbd.list_functions()['Functions'])
lmbd_df.head()

,FunctionName,FunctionArn,Runtime,Role,Handler,CodeSize,Description,Timeout,MemorySize,LastModified,CodeSha256,Version,TracingConfig,RevisionId,Layers,PackageType,Architectures,EphemeralStorage,SnapStart,LoggingConfig
0,daily-new-uploads,arn:aws:lambda:us-east-2:586794463896:function...,python3.13,arn:aws:iam::586794463896:role/super_lambda_role,lambda_function.lambda_handler,942,,10,128,2025-02-17T15:28:49.000+0000,Hvawj19YSIc/L41B6iX0aLuNiiKUBHFWEJei8WhKtEA=,$LATEST,{'Mode': 'PassThrough'},b479a00f-f365-47ea-b37b-a4fe6a393567,[{'Arn': 'arn:aws:lambda:us-east-2:33639294834...,Zip,[x86_64],{'Size': 512},"{'ApplyOn': 'None', 'OptimizationStatus': 'Off'}","{'LogFormat': 'Text', 'LogGroup': '/aws/lambda..."
1,manage_ec2_server,arn:aws:lambda:us-east-2:586794463896:function...,python3.13,arn:aws:iam::586794463896:role/super_lambda_role,lambda_function.lambda_handler,355,,3,128,2025-04-09T16:28:01.000+0000,ePm6VJTAY6KbyVK2E+XO5e8kkMyqMCqLGaS8Y35sMy4=,$LATEST,{'Mode': 'PassThrough'},8bc02dc4-270f-41a7-815a-a567e25ad1af,NaN,Zip,[x86_64],{'Size': 512},"{'ApplyOn': 'None', 'OptimizationStatus': 'Off'}","{'LogFormat': 'Text', 'LogGroup': '/aws/lambda..."


In [121]:
arn = lmbd_df[lmbd_df['FunctionName'] == 'manage_ec2_server'].iloc[0,1]

In [264]:
action_json = json.dumps({
    'instance_id': coop_server
    ,'action': 'stop'
})

In [265]:
lambda_respnse = lmbd.invoke(FunctionName = arn, Payload = action_json)

In [266]:
lambda_respnse

{'ResponseMetadata': {'RequestId': '7979d146-f882-4644-b306-fff036c08276',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 09 Apr 2025 19:35:27 GMT',
   'content-type': 'application/json',
   'content-length': '3',
   'connection': 'keep-alive',
   'x-amzn-requestid': '7979d146-f882-4644-b306-fff036c08276',
   'x-amzn-remapped-content-length': '0',
   'x-amz-executed-version': '$LATEST',
   'x-amzn-trace-id': 'Root=1-67f6cbfe-60457ca50c1b601261a48f95;Parent=36769b5898cc73bf;Sampled=0;Lineage=1:5f02ad21:0'},
  'RetryAttempts': 0},
 'StatusCode': 200,
 'ExecutedVersion': '$LATEST',
 'Payload': <botocore.response.StreamingBody at 0x11656c2e0>}

In [267]:
request_id = lambda_respnse['ResponseMetadata']['HTTPHeaders']['x-amzn-requestid']
request_id

'7979d146-f882-4644-b306-fff036c08276'

In [268]:
api_status_code = lambda_respnse['StatusCode']
api_status_code

200

In [273]:
with open('file.txt','wb') as lambda_response:
    function_status_code = lambda_response.write(lambda_respnse['Payload'].read())


In [274]:
function_status_code

0

In [242]:
get_server_status(coop_server)

'running'

In [243]:
request_id

'd327b798-8760-49c1-b5a9-9b39f00a22f4'

In [262]:
cloud_watch_logs = boto3.client("logs")

log_group = '/aws/lambda/manage_ec2_server'

response = cloud_watch_logs.filter_log_events(
    logGroupName = log_group,
    filterPattern = request_id
)

In [263]:
response

{'events': [],
 'searchedLogStreams': [],
 'nextToken': 'Bxkq6kVGFtq2y_MoigeqscPOdhXVbhiVtLoAmXb5jCoU9c6iPjLV4kp62sffMTHNJ0iW1b-0-AH01jNs93QBwB5VQXf855UjV7noh4Wz7BPCCBUnOzIkcM5vFFK_BnPmUDnQcyxvqCllofxInCDp4T0Dm6X1bK7Kt1EHWBM8dxVgOHr8t-5QyepDTCsPF2pb895mIIBjL-NvYyo4wrWPUD9yzB8GpUBPJX9thnTQF4TU1HigtXtKoBlW0nFh3xUslqUB2E9NpRjjT_w4J4BFBA',
 'ResponseMetadata': {'RequestId': '743750b8-c033-4254-a2ef-101a04269cd5',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '743750b8-c033-4254-a2ef-101a04269cd5',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '414',
   'date': 'Wed, 09 Apr 2025 19:34:41 GMT'},
  'RetryAttempts': 0}}